# Notebook 8 — 4-Tier Final Inference Pipeline & Clinical Report

**This is the final deployed system.** Given any 12-lead ECG (+ optional age/sex/HR) it produces:

```
════════ ECG RISK ASSESSMENT ════════

════════ TIER 1 — Risk Prediction ════════
  Risk Level : HIGH (81%)
  Low: 4%  Moderate: 11%  High: 81%  Critical: 4%

════════ TIER 2 — Explainability ════════
  Lead Importance: V2 28%  V3 23%  II 18%
  Regions: QRS Complex  ST Segment
  Key Timestamps: 1350 ms  1380 ms  1420 ms
  GradCAM Heatmap + IG Saliency + Attention Map

════════ TIER 3 — Clinical Interpretation ════════
  Rhythm: Normal Sinus Rhythm
  Abnormality: Bundle Branch Block
  Clinical Findings: ST-T Wave Changes
  Most Important Leads: V2, V3, II

════════ TIER 4 — Reliability ════════
  Confidence: 91.4%
  Uncertainty: 3%
  Reliability: STRONG
  Action: Cardiology follow-up advised.
══════════════════════════════
```


In [1]:
import torch, sys, json, warnings, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from scipy.signal import resample as scipy_resample
from scipy.ndimage import zoom
from sklearn.metrics import f1_score, classification_report, confusion_matrix
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = Path("./ECG_Project/data")
sys.path.insert(0, str(SAVE_DIR))
from ecg_model import ECGRiskNetXAI

with open(SAVE_DIR / "metadata.json") as f: META = json.load(f)
RISK_LABELS = {int(k): v for k, v in META["risk_labels"].items()}
LEAD_NAMES  = META.get("standard_lead_order",
    ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"])

# ── Load full-mix model (trained in NB6e on all 4 datasets) ──────────────
model = ECGRiskNetXAI(in_ch=12, base_ch=64, meta_dim=3, dropout=0.3).to(device)
model.load_state_dict(torch.load(SAVE_DIR / "fullmix_model.pt", map_location=device))
model.eval()

# Load temperature scaler
class TemperatureScaling(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = torch.nn.Parameter(torch.ones(1) * 1.5)
    def forward(self, logits):
        return logits / self.temperature

temp_scaler = TemperatureScaling()
try:
    temp_scaler.load_state_dict(torch.load(SAVE_DIR / "temperature_scaler.pt", map_location="cpu"))
    print(f"✅ Temperature scaler loaded (T={temp_scaler.temperature.item():.4f}).")
except Exception:
    print("⚠️  Temperature scaler not found — using T=1.0 (uncalibrated).")
temp_scaler.eval()

print("✅ Full-mix model (NB6e) loaded for 4-tier inference.")
print(f"Device: {device}")


✅ Temperature scaler loaded (T=1.4006).
✅ Model loaded for 4-tier inference.
Device: cuda


## Step 1: Clinical Knowledge Tables (Tier 3)

In [2]:
# ── Tier 3 Clinical Labels ────────────────────────────────────────────────────
ARRHYTHMIA_LABELS = [
    "Normal Sinus Rhythm",
    "Atrial Fibrillation",
    "Bradycardia",
    "Tachycardia",
    "Premature Ventricular Contractions",
    "Other Arrhythmia",
]

MI_LABELS = ["No MI Detected", "MI Pattern Present"]

STT_LABELS = [
    "Normal ST-T",
    "ST Elevation",
    "ST Depression / T-wave Abnormality",
]

CONDUCTION_LABELS = [
    "Normal Conduction",
    "Left Bundle Branch Block (LBBB)",
    "Right Bundle Branch Block (RBBB)",
    "AV Block",
]

# ── Tier 1 Clinical Context per Risk Level ────────────────────────────────────
RISK_EMOJIS = {0: "🟢", 1: "🟡", 2: "🟠", 3: "🔴"}

RISK_DETECTION_REGIONS = {
    0: "Normal waveform morphology",
    1: "P-wave and ST-segment",
    2: "QRS complex and ST-segment",
    3: "ST-segment, QRS, and T-wave",
}

RISK_DISEASES = {
    0: "No significant abnormality",
    1: "Mild arrhythmia / conduction abnormality",
    2: "Ventricular ectopy / Bundle Branch Block / Ischemia",
    3: "Atrial Fibrillation / Myocardial Infarction / Ventricular Tachycardia",
}

RISK_REASONS = {
    0: "ECG pattern consistent with normal sinus rhythm. No acute changes detected.",
    1: "Mild cardiac abnormality detected. Non-urgent but warrants clinical review.",
    2: "Significant cardiac abnormality. Prompt clinical evaluation recommended.",
    3: "Critical cardiac pattern identified. Immediate clinical attention required.",
}

RISK_ACTIONS = {
    0: "Routine follow-up. No urgent action required.",
    1: "Schedule non-urgent cardiology review within 1–2 weeks.",
    2: "Urgent cardiology evaluation recommended within 24–48 hours.",
    3: "Immediate medical attention. Activate emergency protocol if indicated.",
}

print("✅ Clinical knowledge tables loaded.")


✅ Clinical knowledge tables loaded.


## Step 2: Tier 4 — Monte Carlo Dropout Uncertainty Estimation

Runs 50 stochastic forward passes with dropout enabled to estimate prediction uncertainty.


In [3]:
def mc_dropout_predict(model, ecg_t, meta_t, n_passes=50):
    """
    Monte Carlo Dropout uncertainty estimation.
    Enables dropout during inference and runs n_passes forward passes.

    Returns:
        mean_probs  : (4,) mean probability over passes
        uncertainty : scalar — mean predictive entropy (higher = more uncertain)
        reliability : str — STRONG / MODERATE / WEAK
    """
    # Enable dropout (train mode for MC Dropout, but no grad)
    model.train()   # dropout active

    all_probs = []
    with torch.no_grad():
        for _ in range(n_passes):
            out   = model(ecg_t, meta_t)
            probs = torch.softmax(out["risk"], dim=1).squeeze(0).cpu().numpy()
            all_probs.append(probs)

    model.eval()   # restore eval mode

    all_probs  = np.stack(all_probs)          # (n_passes, 4)
    mean_probs = all_probs.mean(axis=0)       # (4,)

    # Predictive entropy: H = -sum(p * log(p))
    entropy    = -np.sum(mean_probs * np.log(mean_probs + 1e-8))
    max_entropy = np.log(4)                   # maximum entropy for 4 classes
    uncertainty = float(entropy / max_entropy)  # normalised [0, 1]

    if uncertainty < 0.15:
        reliability = "STRONG"
    elif uncertainty < 0.35:
        reliability = "MODERATE"
    else:
        reliability = "WEAK"

    return mean_probs, uncertainty, reliability


print("✅ MC Dropout uncertainty estimator defined.")


✅ MC Dropout uncertainty estimator defined.


## Step 3: Full 4-Tier Inference Pipeline

In [ ]:
def predict_ecg_4tier(
    signal_1000x12,
    age_years=60.0,
    sex=1,
    heart_rate=75.0,
    fs=100,
    mc_passes=50,
):
    """
    Full 4-Tier inference pipeline.

    Parameters
    ----------
    signal_1000x12 : np.ndarray (1000, 12) — 12-lead ECG, 10 s @ 100 Hz
    age_years      : float  — patient age in years (default 60)
    sex            : int    — 0=Female, 1=Male (default 1)
    heart_rate     : float  — heart rate in BPM (default 75)
    fs             : int    — sampling freq of input (resampled to 100Hz if different)
    mc_passes      : int    — MC Dropout passes for Tier 4 uncertainty

    Returns
    -------
    dict with all 4 tiers of output
    """
    signal = signal_1000x12.astype(np.float32)

    # ── Preprocess ────────────────────────────────────────────────────────────
    if signal.shape[0] != 1000:
        signal = scipy_resample(signal, 1000, axis=0)

    mean = np.mean(signal, axis=0, keepdims=True)
    std  = np.where(np.std(signal, axis=0, keepdims=True) < 1e-8, 1e-8,
                    np.std(signal, axis=0, keepdims=True))
    signal = (signal - mean) / std
    signal = np.nan_to_num(signal, nan=0.0, posinf=0.0, neginf=0.0)

    # ── Tensor preparation ────────────────────────────────────────────────────
    ecg_t  = torch.from_numpy(signal.T).unsqueeze(0).float().to(device)   # (1, 12, 1000)
    meta_np = np.array([
        np.clip(age_years, 0, 100) / 100.0,
        float(sex),
        np.clip(heart_rate, 30, 250) / 200.0,
    ], dtype=np.float32)
    meta_t = torch.from_numpy(meta_np).unsqueeze(0).float().to(device)   # (1, 3)

    # ── TIER 1: Risk Prediction ───────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        out = model(ecg_t, meta_t)

    # Apply temperature scaling
    logits_cal = temp_scaler(out["risk"].cpu())
    t1_probs   = torch.softmax(logits_cal, dim=1).squeeze(0).detach().cpu().numpy()   # (4,)
    t1_pred    = int(t1_probs.argmax())
    t1_conf    = float(t1_probs[t1_pred])

    # ── TIER 2: Explainability ────────────────────────────────────────────────
    # Attention pooling weights → time importance
    pool_w = model.last_pool_weights
    if pool_w is not None:
        pool_w = pool_w.squeeze().numpy()          # (T,) ≈ 125
        factor = 1000 / len(pool_w)
        attn_importance = zoom(pool_w, factor)     # (1000,)
        attn_importance = ((attn_importance - attn_importance.min()) /
                           (attn_importance.max() - attn_importance.min() + 1e-8))
    else:
        attn_importance = np.ones(1000) / 1000

    # SE block → lead importance (proxy via gradient)
    ecg_t2 = ecg_t.clone().detach().requires_grad_(True)
    out2   = model(ecg_t2, meta_t)
    out2["risk"][0, t1_pred].backward()
    lead_imp = ecg_t2.grad.abs().mean(dim=-1).squeeze(0).cpu().numpy()   # (12,)
    lead_imp = (lead_imp - lead_imp.min()) / (lead_imp.max() - lead_imp.min() + 1e-8)

    # Top leads
    top_lead_idx  = lead_imp.argsort()[::-1][:5]
    top_lead_names = [(LEAD_NAMES[i], float(lead_imp[i])) for i in top_lead_idx]

    # Key timestamps (top 5% of attention)
    threshold    = np.percentile(attn_importance, 95)
    hot_samples  = np.where(attn_importance >= threshold)[0]
    positions_ms = sorted(set((hot_samples / 100 * 1000).astype(int).tolist()))[:10]

    # ECG region detection (simple heuristic over 10s window @ 100 Hz)
    region_labels = []
    hot_set = set(hot_samples)
    # Typical P-wave: 0-200 ms, QRS: 200-280 ms, ST: 280-400 ms, T: 400-600 ms
    if any(h < 200 for h in hot_samples): region_labels.append("P-wave")
    if any(200 <= h < 280 for h in hot_samples): region_labels.append("QRS Complex")
    if any(280 <= h < 400 for h in hot_samples): region_labels.append("ST Segment")
    if any(400 <= h < 600 for h in hot_samples): region_labels.append("T-wave")
    if not region_labels: region_labels = ["QRS Complex"]

    # ── TIER 3: Clinical Interpretation ──────────────────────────────────────
    with torch.no_grad():
        out3 = model(ecg_t, meta_t)

    arrhythmia_pred = int(torch.softmax(out3["arrhythmia"], 1).argmax(1).item())
    mi_pred         = int(torch.softmax(out3["mi"],         1).argmax(1).item())
    st_t_pred       = int(torch.softmax(out3["st_t"],       1).argmax(1).item())
    conduction_pred = int(torch.softmax(out3["conduction"], 1).argmax(1).item())

    # ── TIER 4: Reliability (MC Dropout) ─────────────────────────────────────
    mc_probs, uncertainty, reliability = mc_dropout_predict(
        model, ecg_t, meta_t, n_passes=mc_passes
    )
    # Use calibrated single-pass confidence for display
    confidence = float(t1_conf)

    if reliability == "WEAK":
        action_note = " Manual cardiologist review recommended due to high uncertainty."
    else:
        action_note = ""

    return {
        # TIER 1
        "t1_risk_level"     : t1_pred,
        "t1_risk_label"     : RISK_LABELS[t1_pred],
        "t1_confidence"     : confidence,
        "t1_probabilities"  : {RISK_LABELS[i]: float(t1_probs[i]) for i in range(4)},
        # TIER 2
        "t2_attn_importance" : attn_importance,
        "t2_lead_importance" : lead_imp,
        "t2_top_leads"       : top_lead_names,
        "t2_regions"         : region_labels,
        "t2_timestamps_ms"   : positions_ms,
        # TIER 3
        "t3_rhythm"          : ARRHYTHMIA_LABELS[arrhythmia_pred],
        "t3_mi"              : MI_LABELS[mi_pred],
        "t3_st_t"            : STT_LABELS[st_t_pred],
        "t3_conduction"      : CONDUCTION_LABELS[conduction_pred],
        "t3_detection_region": RISK_DETECTION_REGIONS[t1_pred],
        "t3_disease"         : RISK_DISEASES[t1_pred],
        "t3_reason"          : RISK_REASONS[t1_pred],
        # TIER 4
        "t4_confidence"      : confidence,
        "t4_uncertainty"     : uncertainty,
        "t4_reliability"     : reliability,
        "t4_action"          : RISK_ACTIONS[t1_pred] + action_note,
        # Raw
        "signal"             : signal,
    }


print("✅ predict_ecg_4tier() pipeline ready.")


✅ predict_ecg_4tier() pipeline ready.


## Step 4: Clinical Report Generator

In [5]:
def print_clinical_report(result):
    """Print the full 4-tier clinical report to stdout."""
    rl  = result["t1_risk_label"].upper()
    ico = RISK_EMOJIS[result["t1_risk_level"]]
    W   = 54

    print("═" * W)
    print(f"  ECG RISK ASSESSMENT  —  {ico}  {rl}")
    print("═" * W)

    print("\n═══ TIER 1 — Risk Prediction ═══")
    print(f"  Risk Level  : {ico} {result['t1_risk_label'].upper()}")
    print(f"  Confidence  : {result['t1_confidence']*100:.1f}%")
    print("  Probabilities:")
    for cls, prob in result["t1_probabilities"].items():
        bar = "█" * int(prob * 30)
        print(f"    {cls:10s} {prob*100:5.1f}%  {bar}")

    print("\n═══ TIER 2 — Explainability ═══")
    print("  Lead Importance:")
    for name, imp in result["t2_top_leads"]:
        print(f"    {name:6s}  {imp*100:.1f}%")
    print(f"  ECG Regions : {', '.join(result['t2_regions'])}")
    print(f"  Key Times   : {', '.join(str(t)+' ms' for t in result['t2_timestamps_ms'][:6])}")

    print("\n═══ TIER 3 — Clinical Interpretation ═══")
    print(f"  Rhythm      : {result['t3_rhythm']}")
    print(f"  MI Pattern  : {result['t3_mi']}")
    print(f"  ST/T Status : {result['t3_st_t']}")
    print(f"  Conduction  : {result['t3_conduction']}")
    print(f"  Disease     : {result['t3_disease']}")
    print(f"  Reason      : {result['t3_reason']}")

    print("\n═══ TIER 4 — Reliability ═══")
    rel = result["t4_reliability"]
    rel_icon = {"STRONG": "✅", "MODERATE": "⚠️", "WEAK": "❌"}.get(rel, "")
    print(f"  Confidence  : {result['t4_confidence']*100:.1f}%")
    print(f"  Uncertainty : {result['t4_uncertainty']*100:.1f}%")
    print(f"  Reliability : {rel_icon} {rel}")
    print(f"  Action      : {result['t4_action']}")

    print("\n" + "═" * W)


print("✅ print_clinical_report() defined.")


✅ print_clinical_report() defined.


## Step 5: 4-Panel Visualization

In [6]:
def plot_4tier_result(result, lead_name="Lead II", lead_idx=1, save_path=None):
    """Plot the full 4-tier result in a 6-panel figure."""
    signal    = result["signal"]
    attn_imp  = result["t2_attn_importance"]
    lead      = signal[:, lead_idx]
    t         = np.arange(1000) / 100

    COLORS = {0: "#27AE60", 1: "#2980B9", 2: "#F39C12", 3: "#E74C3C"}
    risk_color = COLORS[result["t1_risk_level"]]
    ico = RISK_EMOJIS[result["t1_risk_level"]]

    fig = plt.figure(figsize=(20, 14))
    gs  = fig.add_gridspec(3, 3, height_ratios=[2, 2, 1.5],
                           hspace=0.45, wspace=0.35)

    # ── Panel 1: Raw ECG ───────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(t, lead, color="black", linewidth=0.7, alpha=0.85)
    ax1.set_title(f"ECG Waveform — {lead_name}  |  TIER 1: "
                  f"{ico} {result['t1_risk_label']}  "
                  f"({result['t1_confidence']*100:.1f}%)",
                  fontsize=12, fontweight="bold", color=risk_color)
    ax1.set_xlabel("Time (s)"); ax1.set_ylabel("Amplitude (norm)")
    ax1.grid(True, alpha=0.2)

    # ── Panel 2: Attention Heatmap (Tier 2) ───────────────────────────────
    ax2 = fig.add_subplot(gs[1, :])
    ax2.plot(t, lead, color="black", linewidth=0.6, alpha=0.5)
    for i in range(len(t) - 1):
        ax2.axvspan(t[i], t[i+1], alpha=float(attn_imp[i]) * 0.7,
                    color=cm.hot(float(attn_imp[i])), linewidth=0)
    for pos_ms in result["t2_timestamps_ms"][:6]:
        ax2.axvline(pos_ms / 1000, color=risk_color,
                    linestyle="--", alpha=0.7, linewidth=1.2)
    ax2.set_title("TIER 2 — Attention Heatmap  |  Red = High Importance  |  "
                  f"Regions: {', '.join(result['t2_regions'])}",
                  fontsize=11, fontweight="bold")
    ax2.set_xlabel("Time (s)"); ax2.set_ylabel("Amplitude")
    ax2.grid(True, alpha=0.15)

    # ── Panel 3: Risk Probability Bar (Tier 1) ────────────────────────────
    ax3 = fig.add_subplot(gs[2, 0])
    cls_list  = list(result["t1_probabilities"].keys())
    prob_list = list(result["t1_probabilities"].values())
    bar_cols  = [COLORS[i] for i in range(4)]
    bars = ax3.barh(cls_list, prob_list, color=bar_cols, edgecolor="white")
    ax3.set_xlim(0, 1.2)
    ax3.set_xlabel("Probability")
    ax3.set_title("TIER 1 — Risk Probabilities", fontweight="bold")
    for bar, prob in zip(bars, prob_list):
        ax3.text(prob + 0.02, bar.get_y() + bar.get_height()/2,
                 f"{prob*100:.1f}%", va="center", fontsize=10)

    # ── Panel 4: Lead Importance (Tier 2) ────────────────────────────────
    ax4 = fig.add_subplot(gs[2, 1])
    lead_imp = result["t2_lead_importance"]
    colors_lead = plt.cm.RdYlGn(1 - lead_imp)
    ax4.bar(LEAD_NAMES, lead_imp * 100, color=colors_lead)
    ax4.set_title("TIER 2 — Lead Importance", fontweight="bold")
    ax4.set_xlabel("Lead"); ax4.set_ylabel("%")
    ax4.tick_params(axis="x", rotation=45, labelsize=8)
    ax4.grid(True, alpha=0.3)

    # ── Panel 5: Clinical Summary (Tier 3 + 4) ───────────────────────────
    ax5 = fig.add_subplot(gs[2, 2])
    ax5.axis("off")
    rel_icon = {"STRONG": "✅", "MODERATE": "⚠️", "WEAK": "❌"}.get(result["t4_reliability"], "")
    summary = (
        f"TIER 3 — Clinical\n"
        f"Rhythm    : {result['t3_rhythm']}\n"
        f"MI        : {result['t3_mi']}\n"
        f"ST/T      : {result['t3_st_t']}\n"
        f"Conduction: {result['t3_conduction']}\n"
        f"Disease   : {result['t3_disease'][:35]}\n\n"
        f"TIER 4 — Reliability\n"
        f"Confidence  : {result['t4_confidence']*100:.1f}%\n"
        f"Uncertainty : {result['t4_uncertainty']*100:.1f}%\n"
        f"Reliability : {rel_icon} {result['t4_reliability']}\n"
        f"Action:\n{result['t4_action'][:60]}"
    )
    ax5.text(0.02, 0.98, summary, va="top", ha="left",
             fontsize=8.5, transform=ax5.transAxes,
             fontfamily="monospace",
             bbox=dict(boxstyle="round", facecolor=risk_color, alpha=0.12))
    ax5.set_title("Tier 3 & 4 Summary", fontweight="bold")

    fig.suptitle(
        f"ECGRiskNet-XAI — 4-Tier Clinical Report  |  "
        f"{ico} {result['t1_risk_label']}  ({result['t1_confidence']*100:.1f}% confident)",
        fontsize=14, fontweight="bold", color=risk_color
    )

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


print("✅ plot_4tier_result() defined.")


✅ plot_4tier_result() defined.


## Step 6: Run on Test Samples — One per Risk Class

In [8]:
# ── Load full-mix pooled test set (NB6e output) ───────────────────────────
def _load_npz(path, key_x="X", key_y="y", key_m="meta"):
    d = np.load(path)
    X = d[key_x].astype(np.float32)
    y = d[key_y].astype(np.int64)
    M = d[key_m].astype(np.float32) if key_m in d else np.zeros((len(y),3),np.float32)
    return X, y, M

X_parts, y_parts, M_parts = [], [], []
for npz_name in ["test_processed.npz", "incart_test.npz", "cpsc2018_test.npz", "chapman_test.npz"]:
    p = SAVE_DIR / npz_name
    if p.exists():
        Xp, yp, Mp = _load_npz(p)
        X_parts.append(Xp); y_parts.append(yp); M_parts.append(Mp)
        print(f"  Loaded {npz_name}: {Xp.shape}")

if len(X_parts) == 0:
    raise FileNotFoundError("No test npz files found under SAVE_DIR.")

X_test = np.concatenate(X_parts, axis=0)
y_test = np.concatenate(y_parts, axis=0)
M_test = np.concatenate(M_parts, axis=0)
print(f"Full-mix pooled test set: {X_test.shape}")

print("Running 4-tier inference — one sample per risk class...\n")

all_results = {}

for cls in range(4):
    idx = np.where(y_test == cls)[0]
    if len(idx) == 0:
        print(f"  No samples for class {cls} ({RISK_LABELS[cls]})"); continue

    sample  = X_test[idx[0]]   # (1000, 12)
    meta_i  = M_test[idx[0]]   # (3,)
    age     = float(meta_i[0] * 100)
    sex     = int(round(meta_i[1]))
    hr      = float(meta_i[2] * 200)

    result = predict_ecg_4tier(
        sample, age_years=age, sex=sex, heart_rate=hr, mc_passes=50
    )
    all_results[cls] = result

    print(f"\n--- TRUE LABEL: {RISK_LABELS[cls]} ---")
    print_clinical_report(result)

    plot_4tier_result(
        result,
        lead_name=f"Lead II  |  True: {RISK_LABELS[cls]}",
        save_path=SAVE_DIR / f"4tier_result_class{cls}.png"
    )


Running 4-tier inference — one sample per risk class...



RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.

## Step 7: Batch Inference on Full Test Set (PTB-XL)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

X_t = torch.from_numpy(X_test.transpose(0, 2, 1))   # (N, 12, 1000)
M_t = torch.from_numpy(M_test)
loader = DataLoader(TensorDataset(X_t, M_t), batch_size=64,
                    shuffle=False, num_workers=0)

all_preds, all_probs = [], []
model.eval()
with torch.no_grad():
    for xb, mb in loader:
        out   = model(xb.to(device), mb.to(device))
        # Apply temperature scaling
        logits_cal = temp_scaler(out["risk"].cpu())
        probs  = torch.softmax(logits_cal, dim=1).detach().cpu().numpy()
        preds  = probs.argmax(axis=1)
        all_preds.extend(preds)
        all_probs.append(probs)

all_preds = np.array(all_preds)
all_probs = np.vstack(all_probs)

acc = (all_preds == y_test).mean()
mf1 = f1_score(y_test, all_preds, average="macro", zero_division=0)

print(f"Final Test Accuracy (PTB-XL)  : {acc*100:.2f}%")
print(f"Final Test Macro F1 (PTB-XL)  : {mf1:.4f}")
print()
print(classification_report(y_test, all_preds,
      target_names=["Low","Moderate","High","Critical"], digits=4, zero_division=0))

# Final confusion matrix
cm = confusion_matrix(y_test, all_preds, labels=[0,1,2,3])
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Low","Moderate","High","Critical"],
            yticklabels=["Low","Moderate","High","Critical"], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Final Confusion Matrix — PTB-XL Test", fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "final_confusion_matrix.png", dpi=150)
plt.show()


## Step 8: Final Project Summary

In [ ]:
# Load saved results
def safe_load(path):
    try:
        with open(path) as f: return json.load(f)
    except Exception:
        return {}

# ── NB6e full-mix results ─────────────────────────────────────────────────
fm = safe_load(SAVE_DIR / "fullmix_training_results.json")
em = safe_load(SAVE_DIR / "edge_metrics.json")

# Helper to pull per-split metrics from fullmix JSON
def fm_get(split_key, metric, default=0):
    return fm.get(split_key, {}).get(metric, default)

print("=" * 60)
print("  ECGRiskNet-XAI — FINAL PROJECT SUMMARY (Full-Mix / NB6e)")
print("=" * 60)

print("\nARCHITECTURE")
print("  Conv Stem → InceptionTime → SE Attention")
print("  → Transformer Encoder (4×) → Attention Pooling")
print("  → Metadata Fusion → SupCon Head + 5 Multi-Task Heads")
print("  Loss: CBFocalLoss + 0.1×SupConLoss + 0.15×AuxFocalLoss")

print("\nTRAINING (Full-Mix: PTB-XL + INCART + CPSC2018 + Chapman)")
print(f"  Best Val Macro-F1 : {fm.get('best_val_macro_f1', 0):.4f}")
print(f"  Best Epoch        : {fm.get('best_epoch', 0)}")
print(f"  Train Samples     : {fm.get('train_samples', 0):,}")
print(f"  Val   Samples     : {fm.get('val_samples', 0):,}")

print("\nTEST RESULTS PER DATASET (NB6e Full-Mix Checkpoint)")
for split_key, label in [
        ("ptbxl_test",       "PTB-XL"),
        ("incart_test",      "INCART"),
        ("cpsc2018_test",    "CPSC2018"),
        ("chapman_test",     "Chapman"),
        ("pooled_mix_test",  "Pooled Mix"),
]:
    acc = fm_get(split_key, "accuracy", 0)
    mf1 = fm_get(split_key, "macro_f1", 0)
    n   = fm_get(split_key, "n_samples", 0)
    print(f"  {label:<12}: Acc={acc*100:.2f}%  MacroF1={mf1:.4f}  n={n:,}")

print("\nEDGE-AI")
if em:
    print(f"  Baseline latency  : {em.get('Original',{}).get('latency_ms',0):.2f} ms")
    print(f"  Quantized latency : {em.get('Quantized',{}).get('latency_ms',0):.2f} ms")
    print(f"  TorchScript latency:{em.get('TorchScript',{}).get('latency_ms',0):.2f} ms")

print("\n4-TIER OUTPUTS")
print("  ✅ Tier 1: Risk prediction (Low/Moderate/High/Critical)")
print("  ✅ Tier 2: GradCAM1D + SmoothGrad + IG + Attention maps + Lead importance")
print("  ✅ Tier 3: Rhythm + MI + ST/T + Conduction + Disease interpretation")
print("  ✅ Tier 4: MC Dropout uncertainty + Reliability score + Action")

print("\nALL NOTEBOOKS COMPLETE ✅")
print("=" * 60)
